In [ ]:
import torch
from transformers import AutoModelForCausalLM

from model_methods.deepseekvl2_methods import DeepseekVLV2Processor, DeepseekVLV2ForCausalLM
from model_methods.deepseekvl2_methods import load_pil_images


# specify the path to the model
model_path = "/data/VLM/deepseek-vl2-small"
vl_chat_processor: DeepseekVLV2Processor = DeepseekVLV2Processor.from_pretrained(model_path)
tokenizer = vl_chat_processor.tokenizer

vl_gpt: DeepseekVLV2ForCausalLM = AutoModelForCausalLM.from_pretrained(model_path, attn_implementation="eager",trust_remote_code=True)
vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()

from PIL import Image
import io
import base64
image1=Image.open("/data/dataset/MSRS/test/vi/00131D.png").convert("RGB")
image2=Image.open("/data/dataset/MSRS/test/ir/00131D.png").convert("RGB")

def pil_image_to_data_url(image: Image.Image, format="PNG"):
    buffered = io.BytesIO()
    image.save(buffered, format=format)  # PNG, JPEG 等
    img_bytes = buffered.getvalue()
    base64_str = base64.b64encode(img_bytes).decode("utf-8")
    data_url = f"data:image/{format.lower()};base64,{base64_str}"
    return data_url
data_url1 = pil_image_to_data_url(image1)
data_url2 = pil_image_to_data_url(image2)

# multiple images/interleaved image-text
conversation = [
    {
        "role": "<|User|>",
        "content": "This is image_1: <image>\n"
                   "This is image_2: <image>\n"
                   "Can you tell me what are in the images?",
        "images": [
            data_url1,data_url2,
        ],
    },
    {"role": "<|Assistant|>", "content": ""}
]

# load images and prepare for inputs
# pil_images = load_pil_images(conversation)
pil_images=[image1,image2]

prepare_inputs = vl_chat_processor(
    conversations=conversation,
    images=pil_images,
    force_batchify=True,
    system_prompt=""
).to(vl_gpt.device)

# run image encoder to get the image embeddings
inputs_embeds = vl_gpt.prepare_inputs_embeds(**prepare_inputs)



In [ ]:
count = prepare_inputs["input_ids"][0].tolist().count(128815)
count

In [ ]:
def find_zero_blocks(lst, token, min_length=2):
    blocks = []
    i = 0
    while i < len(lst):
        if lst[i] == token:
            start = i
            while i < len(lst) and lst[i] == token:
                i += 1
            if i - start >= min_length:
                blocks.append(start)
        else:
            i += 1
    return blocks

In [ ]:
IMAGE_TOKEN_INDEX=128815
NUM_IMG_TOKENS = 1024
NUM_PATCHES = 32
inputs=prepare_inputs["input_ids"][0].tolist()
poses=find_zero_blocks(inputs,IMAGE_TOKEN_INDEX)


In [ ]:
outputs = vl_gpt.language(
    inputs_embeds=inputs_embeds,
    attention_mask=prepare_inputs.attention_mask,
    use_cache=True,
    output_attentions=True
)

In [ ]:
outputs["attentions"][0].shape

In [ ]:
# # run the model to get the response
# outputs = vl_gpt.language.generate(
#     inputs_embeds=inputs_embeds,
#     attention_mask=prepare_inputs.attention_mask,
#     pad_token_id=tokenizer.eos_token_id,
#     bos_token_id=tokenizer.bos_token_id,
#     eos_token_id=tokenizer.eos_token_id,
#     max_new_tokens=512,
#     do_sample=False,
#     use_cache=True
# )

# answer = tokenizer.decode(outputs[0].cpu().tolist(), skip_special_tokens=False)
# print(f"{prepare_inputs['sft_format'][0]}", answer)